In [ ]:
import importlib
import gc
from astropy.io import fits
import numpy as np
# Сначала импортируем модуль
import velocity_analysis
# ПЕРЕЗАГРУЗКА МОДУЛЯ
importlib.reload(velocity_analysis)
from velocity_analysis import run_velocity_analysis, run_gradient_analysis, run_azimuthal_analysis, plot_image_with_contour, save_to_pdf, close_all_fits_files

# Параметры галактики
galaxy_name = "NGC1566"
telescop = "VLT_MUSE"
dist = 21.3
pa = 222 - 180
incl = 32
shift_vel = 29
r = 4
bound = 0.05
scale_factor = None    # None or int

# Пути к файлам
velocity_file = f"data/input{galaxy_name}/{galaxy_name}_vel_Ha.fits"
rad_file = f"data/input{galaxy_name}/{galaxy_name}_rad.npy"
circ_vel_file = f"data/input{galaxy_name}/{galaxy_name}_vel.npy"
flux_file = f"data/input{galaxy_name}/F_Ha.fits"
sigma_file = f"data/input{galaxy_name}/{galaxy_name}_sigma_Ha.fits"

@save_to_pdf(f"pic/{galaxy_name}_{telescop}R{r}_{bound}_{scale_factor}analysis.pdf")
def run_full_analysis():

    close_all_fits_files()
    # Шаг 1: Анализ скоростей
    resid, model_vel_map, hdu = run_velocity_analysis(
        galaxy_name=galaxy_name,
        telescop=telescop,
        dist=dist,
        pa=pa,
        incl=incl,
        velocity_file=velocity_file,
        # sigma_file=sigma_file,
        rad_file=rad_file,
        smooth=scale_factor,
        circ_vel_file=circ_vel_file,
        flux_file=flux_file,
        shift_vel=shift_vel,
        plot=True,
        save=True
    )

    # Шаг 2: Анализ градиентов
    # Теперь у нас в папке data/{galaxy_name} есть необходимые файлы: 
    # {galaxy_name}_residuals.fits, {galaxy_name}_distance_map.fits, {galaxy_name}_angle_map.fits

    # Формируем пути к сохраненным файлам
    import os
    data_dir = f"data/{galaxy_name}"
    velocity_residuals_file = os.path.join(data_dir, f"{galaxy_name}_{telescop}_residuals.fits")
    distance_map_file = os.path.join(data_dir, f"{galaxy_name}_{telescop}_distance_map.fits")
    angle_map_file = os.path.join(data_dir, f"{galaxy_name}_{telescop}_angle_map.fits")

    grad_xy, grad_r, grad_phi = run_gradient_analysis(
        galaxy_name=galaxy_name,
        telescop=telescop,
        dist=dist,
        velocity_file=velocity_residuals_file,
        distance_file=distance_map_file,
        angle_file=angle_map_file,
        # sigma_file=sigma_file,
        PA = pa,
        plot=True,
        save=True, 
        shift_vel=shift_vel
    )

    # Шаг 3: Азимутальное сканирование
    # Теперь у нас есть градиенты, сохраненные в той же папке
    grad_r_file = os.path.join(data_dir, f"{galaxy_name}_{telescop}_vel_grad_r.fits")
    grad_phi_file = os.path.join(data_dir, f"{galaxy_name}_{telescop}_vel_grad_phi.fits")

    run_azimuthal_analysis(
        galaxy_name=galaxy_name,
        telescop=telescop,
        dist=dist,
        flux_file=flux_file,
        grad_r_file=grad_r_file,
        grad_phi_file=grad_phi_file,
        angle_file=angle_map_file,
        distance_file=distance_map_file,
        R=r,  # пример радиуса
        bound=bound
    )


    plot_image_with_contour(flux_file, distance_map_file, r, bound=bound, percent=99)

    # Принудительный сбор мусора (может помочь)
    gc.collect()

    # Проверка открытых файлов (если нужно)
    for obj in gc.get_objects():
        if isinstance(obj, fits.HDUList):
            print(f"Найден открытый HDUList: {obj}")
            obj.close()

run_full_analysis()


In [ ]:

def close_all_fits_files():
    """Принудительно закрывает все открытые FITS файлы"""
    for obj in gc.get_objects():
        if isinstance(obj, fits.HDUList):
            try:
                obj.close()
                print("Закрыт открытый FITS файл")
            except:
                pass

# Вызовите эту функцию перед открытием файла
close_all_fits_files()

In [ ]:
! fitsheader data/inputNGC1566/NGC1566_sigma_Ha.fits

In [ ]:
import numpy as np

# Открываем файл
data = np.load('data/inputNGC4303/NGC4303_r_Ha.npy')

# Выводим информацию о данных
print("Форма массива:", data.shape)
print("Тип данных:", data.dtype)
print("Размер:", data.size)
print("\nДанные:")
print(data)

In [ ]:
# Принудительный сбор мусора (может помочь)
gc.collect()

# Проверка открытых файлов (если нужно)
for obj in gc.get_objects():
    if isinstance(obj, fits.HDUList):
        print(f"Найден открытый HDUList: {obj}")
        obj.close()

____________________________________________
ZOOM with overlay

In [ ]:
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import velocity_analysis
# ПЕРЕЗАГРУЗКА МОДУЛЯ
importlib.reload(velocity_analysis)
from velocity_analysis import overlay_multiple_fits, plot_with_zoom, save_all_figures

# Использование:
galaxy_name = "NGC1566"
zoom_area1 = [500, 500, 650, 600]
zoom_area2 = [550, 700, 700, 800]
zoom_area3 = [270, 250, 370, 350]
# Замените на пути к вашим файлам
file_ref = f"data/{galaxy_name}/{galaxy_name}_VLT_MUSE_vel_grad_xy.fits"
files = [ f"data/input{galaxy_name}/hlsp_phangs-jwst_jwst_miri_ngc1566_f770w_v1p0_img.fits", f"data/input{galaxy_name}/NGC_1566_I_FUV_g2006.fits"]
filename = f"pic/{galaxy_name}_zoom3.pdf"

with PdfPages(filename) as pdf:

# Совмещаем изображения
    overlay_multiple_fits(file_ref, files, zoom=zoom_area3, pdf=pdf, percents=[0, 55, 10], alphas=[1, 0.9, 0.7], title=r"$V_{xy}$ + dust + UV", galaxy_name=galaxy_name)
    plot_with_zoom(fits.open(file_ref)[0].data, zoom_area=zoom_area3, pdf=pdf)



___________
Dispersion

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_gradient_vs_dispersion(gradient_xy, dispersion, xlabel='Gradient XY', ylabel='Dispersion'):
    """
    Строит scatter plot градиента по XY vs дисперсия
    
    Parameters:
    -----------
    gradient_xy : ndarray
        2D массив градиента по XY
    dispersion : ndarray
        2D массив дисперсии
    xlabel, ylabel : str
        Подписи осей
    """
    
    # Преобразуем 2D массивы в 1D для scatter plot
    grad_flat = gradient_xy.flatten()
    disp_flat = dispersion.flatten()
    
    # Убираем NaN и бесконечные значения
    mask = (np.isfinite(grad_flat) & 
            np.isfinite(disp_flat) & 
            (disp_flat <= 500))
    grad_clean = grad_flat[mask]
    disp_clean = disp_flat[mask]
    
    print(f"Всего точек: {len(grad_clean)}")
    print(f"Gradient range: [{np.min(grad_clean):.3f}, {np.max(grad_clean):.3f}]")
    print(f"Dispersion range: [{np.min(disp_clean):.3f}, {np.max(disp_clean):.3f}]")
    
    # Создаем scatter plot
    plt.figure(figsize=(8, 6))
    
    # Простой scatter
    plt.scatter(grad_clean, disp_clean, alpha=0.6, s=1, c='blue')
    
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f'Scatter: {xlabel} vs {ylabel}')
    plt.grid(True, alpha=0.3)
    
    plt.plot([0, 100], [0,100])
    # Добавляем линии средних
    # plt.axvline(np.mean(grad_clean), color='red', linestyle='--', alpha=0.8, label=f'Mean X: {np.mean(grad_clean):.3f}')
    # plt.axhline(np.mean(disp_clean), color='green', linestyle='--', alpha=0.8, label=f'Mean Y: {np.mean(disp_clean):.3f}')
    
    plt.legend()
    plt.tight_layout()
    plt.show()

# Использование
# plot_gradient_vs_dispersion(gradient_array, dispersion_array)

In [ ]:


def read_gradient_dispersion_fits(gradient_file, dispersion_file, hdu_index=0):
    """
    Читает градиент и дисперсию из FITS-файлов
    
    Parameters:
    -----------
    gradient_file : str
        Путь к FITS-файлу с градиентом
    dispersion_file : str
        Путь к FITS-файлу с дисперсией
    hdu_index : int, optional
        Индекс HDU для чтения (по умолчанию 0)
    
    Returns:
    --------
    gradient_data : ndarray
        Массив градиента
    dispersion_data : ndarray
        Массив дисперсии
    """
    
    print(f"Чтение градиента из: {gradient_file}")
    with fits.open(gradient_file) as hdul:
        gradient_data = hdul[hdu_index].data
        gradient_header = hdul[hdu_index].header
        print(f"  Размер: {gradient_data.shape}")
        print(f"  Тип данных: {gradient_data.dtype}")
    
    print(f"Чтение дисперсии из: {dispersion_file}")
    with fits.open(dispersion_file) as hdul:
        dispersion_data = hdul[hdu_index].data
        dispersion_header = hdul[hdu_index].header
        print(f"  Размер: {dispersion_data.shape}")
        print(f"  Тип данных: {dispersion_data.dtype}")
    
    # Проверяем совпадение размеров
    if gradient_data.shape != dispersion_data.shape:
        print(f"⚠️ ВНИМАНИЕ: Размеры не совпадают!")
        print(f"  Градиент: {gradient_data.shape}")
        print(f"  Дисперсия: {dispersion_data.shape}")
        
        # Автоматическое обрезание до минимального размера
        min_shape = (min(gradient_data.shape[0], dispersion_data.shape[0]),
                    min(gradient_data.shape[1], dispersion_data.shape[1]))
        
        gradient_data = gradient_data[:min_shape[0], :min_shape[1]]
        dispersion_data = dispersion_data[:min_shape[0], :min_shape[1]]
        print(f"  Обрезано до: {min_shape}")
    
    return gradient_data, dispersion_data, gradient_header, dispersion_header

In [ ]:
grad, sigma, _, h = read_gradient_dispersion_fits("data/NGC1566/NGC1566_VLT_MUSE_vel_grad_xy.fits", "data/inputNGC1566/NGC1566_sigma_Ha.fits")
plot_gradient_vs_dispersion(grad, sigma)

In [ ]:
def verify_halpha_dispersion_units(dispersion_data, header):
    """
    Проверяет, в км/ли дисперсия Hα
    """
    print("=== ПРОВЕРКА ДИСПЕРСИИ Hα ===")
    
    # Проверяем заголовок
    if 'BUNIT' in header:
        bunit = header['BUNIT'].upper()
        print(f"BUNIT: {bunit}")
        
        if any(unit in bunit for unit in ['KM/S', 'KM S', 'KM/SEC']):
            print("✓ Данные уже в км/с")
            return True
        else:
            print(f"⚠ Нестандартные единицы: {bunit}")
    
    # Проверяем разумный диапазон для дисперсии скоростей
    data_min = np.nanmin(dispersion_data)
    data_max = np.nanmax(dispersion_data)
    
    print(f"Диапазон значений: [{data_min:.1f}, {data_max:.1f}] км/с")
    
    # Типичный диапазон для дисперсии скоростей в галактиках
    if 0 <= data_min <= data_max <= 500:
        print("✓ Диапазон соответствует дисперсии скоростей в км/с")
        return True
    elif data_max > 1000:
        print("⚠ Слишком большие значения - возможно данные в других единицах")
        return False
    else:
        print("✓ Вероятно данные в км/с")
        return True

# Использование
is_kms = verify_halpha_dispersion_units(sigma, h)

In [ ]:
def show_instrumental_limitations():
    """Демонстрирует ограничения инструмента MUSE"""
    
    # Характеристики MUSE
    spectral_resolution = 3000  # Типичное разрешение MUSE
    wavelength = 6562.8  # Длина волны Hα в Å
    
    # Расчет инструментального уширения
    instrumental_fwhm = wavelength / spectral_resolution  # в Å
    instrumental_sigma = instrumental_fwhm / 2.355  # σ в Å
    
    # Перевод в км/с
    instrumental_sigma_kms = instrumental_sigma / wavelength * 299792.458
    
    print("=== ИНСТРУМЕНТАЛЬНЫЕ ОГРАНИЧЕНИЯ MUSE ===")
    print(f"Спектральное разрешение: R = {spectral_resolution}")
    print(f"Длина волны Hα: {wavelength} Å")
    print(f"Инструментальная FWHM: {instrumental_fwhm:.2f} Å")
    print(f"Инструментальная σ: {instrumental_sigma:.2f} Å")
    print(f"Инструментальная σ в км/с: {instrumental_sigma_kms:.1f} км/с")
    
    # Реальная ширина линии = √(σ_obs² - σ_inst²)
    observed_min = 48.9  # Ваше минимальное значение
    intrinsic_min = np.sqrt(observed_min**2 - instrumental_sigma_kms**2)
    
    print(f"\nВаше минимальное наблюдение: {observed_min:.1f} км/с")
    print(f"Внутренняя дисперсия (после коррекции): {intrinsic_min:.1f} км/с")
    
    return instrumental_sigma_kms

instrumental_sigma = show_instrumental_limitations()

In [ ]:
def analyze_high_dispersion_regions(dispersion_data, gradient_data, flux_data=None, threshold=100):
    """
    Анализирует регионы с дисперсией > 100 км/с
    """
    
    dispersion_flat = dispersion_data.flatten()
    gradient_flat = gradient_data.flatten()
    
    # Маски для разных уровней дисперсии
    mask_low = dispersion_flat <= 80
    mask_medium = (dispersion_flat > 80) & (dispersion_flat <= 150)
    mask_high = dispersion_flat > 150
    mask_very_high = dispersion_flat > 200
    
    mask_finite = np.isfinite(dispersion_flat) & np.isfinite(gradient_flat)
    
    print("=== АНАЛИЗ ОБЛАСТЕЙ С ВЫСОКОЙ ДИСПЕРСИЕЙ ===")
    print(f"Всего пикселей: {np.sum(mask_finite):,}")
    print(f"Дисперсия ≤ 80 км/с: {np.sum(mask_low & mask_finite):,} ({np.sum(mask_low & mask_finite)/np.sum(mask_finite)*100:.1f}%)")
    print(f"Дисперсия 80-150 км/с: {np.sum(mask_medium & mask_finite):,} ({np.sum(mask_medium & mask_finite)/np.sum(mask_finite)*100:.1f}%)")
    print(f"Дисперсия > 150 км/с: {np.sum(mask_high & mask_finite):,} ({np.sum(mask_high & mask_finite)/np.sum(mask_finite)*100:.1f}%)")
    print(f"Дисперсия > 200 км/с: {np.sum(mask_very_high & mask_finite):,} ({np.sum(mask_very_high & mask_finite)/np.sum(mask_finite)*100:.1f}%)")
    
    # Визуализация
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 1. Распределение по дисперсии
    axes[0,0].hist(dispersion_flat[mask_finite], bins=50, alpha=0.7)
    axes[0,0].axvline(100, color='red', linestyle='--', label='100 км/с')
    axes[0,0].axvline(150, color='orange', linestyle='--', label='150 км/с')
    axes[0,0].axvline(200, color='red', linestyle='--', label='200 км/с')
    axes[0,0].axvline(np.mean(dispersion_flat[mask_finite]), color='green', linestyle='-', label=f'Mean {np.mean(dispersion_flat[mask_finite])} км/с')
    # plt.axhline(np.mean(disp_clean), color='green', linestyle='--', alpha=0.8, label=f'Mean Y: {np.mean(disp_clean):.3f}')

    axes[0,0].set_xlabel('Дисперсия (км/с)')
    axes[0,0].set_ylabel('Количество пикселей')
    axes[0,0].set_title('Распределение дисперсии')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Scatter plot с цветовой кодировкой
    colors = ['green', 'orange', 'red']
    labels = [f'{np.sum(mask_low & mask_finite)} pix (≤80 km/s)', f'{np.sum(mask_medium & mask_finite)} pix (80-150 km/s)', f'{np.sum(mask_high & mask_finite & mask_very_high)} pix (>150 km/s)']
    masks = [mask_low, mask_medium, mask_high]
    
    for i, (mask_level, color, label) in enumerate(zip(masks, colors, labels)):
        current_mask = mask_level & mask_finite
        axes[0,1].scatter(gradient_flat[current_mask], dispersion_flat[current_mask], 
                         alpha=0.6, s=1, color=color, label=label)
    
    axes[0,1].set_xlabel('Градиент скорости')
    axes[0,1].set_ylabel('Дисперсия (км/с)')
    axes[0,1].set_title('Дисперсия vs Градиент')
    axes[0,1].legend(markerscale=3)
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Процент высокодисперсных регионов
    categories = ['≤80 км/с', '80-150 км/с', '>150 км/с']
    counts = [np.sum(mask_low & mask_finite), np.sum(mask_medium & mask_finite), np.sum(mask_high & mask_finite)]
    
    axes[0,2].pie(counts, labels=categories, colors=colors, autopct='%1.1f%%')
    axes[0,2].set_title('Распределение по уровням дисперсии')
    
    # 4. Статистика по градиентам для разных уровней дисперсии
    dispersion_levels = [
        ('Низкая (≤80)', mask_low),
        ('Средняя (80-150)', mask_medium), 
        ('Высокая (>150)', mask_high)
    ]
    
    grad_means = []
    grad_stds = []
    
    for label, mask in dispersion_levels:
        current_mask = mask & mask_finite
        grad_means.append(np.mean(gradient_flat[current_mask]))
        grad_stds.append(np.std(gradient_flat[current_mask]))
    
    x_pos = np.arange(len(dispersion_levels))
    axes[1,0].bar(x_pos, grad_means, yerr=grad_stds, capsize=5, color=colors, alpha=0.7)
    axes[1,0].set_xticks(x_pos)
    axes[1,0].set_xticklabels([label for label, _ in dispersion_levels])
    axes[1,0].set_ylabel('Средний градиент')
    axes[1,0].set_title('Градиенты по уровням дисперсии')
    axes[1,0].grid(True, alpha=0.3)
    
    # 5. Пространственное распределение (если есть координаты)
    if flux_data is not None:
        flux_flat = flux_data.flatten()
        mask_flux_finite = mask_finite & np.isfinite(flux_flat)
        
        axes[1,1].scatter(flux_flat[mask_flux_finite], dispersion_flat[mask_flux_finite], 
                         alpha=0.3, s=1, c=dispersion_flat[mask_flux_finite], cmap='viridis')
        axes[1,1].set_xlabel('Поток Hα')
        axes[1,1].set_ylabel('Дисперсия (км/с)')
        axes[1,1].set_title('Дисперсия vs Поток')
        axes[1,1].set_yscale('log')
        axes[1,1].set_xscale('log')
        axes[1,1].grid(True, alpha=0.3)
    
    # 6. Кумулятивное распределение
    sorted_disp = np.sort(dispersion_flat[mask_finite])
    y_cumulative = np.arange(1, len(sorted_disp)+1) / len(sorted_disp)
    axes[1,2].plot(sorted_disp, y_cumulative * 100, linewidth=2)
    axes[1,2].axvline(100, color='red', linestyle='--', alpha=0.7, label='100 км/с')
    axes[1,2].axvline(150, color='orange', linestyle='--', alpha=0.7, label='150 км/с')
    axes[1,2].set_xlabel('Дисперсия (км/с)')
    axes[1,2].set_ylabel('Кумулятивный процент (%)')
    axes[1,2].set_title('Кумулятивное распределение')
    axes[1,2].legend()
    axes[1,2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'low_disp': (gradient_flat[mask_low & mask_finite], dispersion_flat[mask_low & mask_finite]),
        'medium_disp': (gradient_flat[mask_medium & mask_finite], dispersion_flat[mask_medium & mask_finite]),
        'high_disp': (gradient_flat[mask_high & mask_finite], dispersion_flat[mask_high & mask_finite])
    }

In [ ]:
with fits.open(flux_file) as hdul:
    # Показываем информацию о всех HDU
    hdul.info()
    
    # Берем данные из первого HDU с данными
    for i, hdu in enumerate(hdul):
        if hdu.data is not None:
            print(f"HDU[{i}]: форма {hdu.data.shape}")
            flux_data = hdu.data
            flux_header = hdu.header
            break

In [ ]:
analyze_high_dispersion_regions(sigma, grad, flux_data)

______________________________________________________________________________________________
SUPERNOVAe

In [26]:
import importlib
# Сначала импортируем модуль
import velocity_analysis
# ПЕРЕЗАГРУЗКА МОДУЛЯ
importlib.reload(velocity_analysis)
from velocity_analysis import plot_supernovae_on_image, plot_supernovae_on_image_zoom


# Пример использования:
if __name__ == "__main__":
    
    # Затем основную функцию
    plot_supernovae_on_image(
        fits_file='data/NGC1566/NGC1566_ALMA_vel_grad_xy.fits',
        sn_file='data/inputNGC1566/SN.txt',
        output_file="pic/trash/supernovae_map_ALMA.png"
    )

d:\vs_code\vel_grad\velocity_gradients\velocity_analysis.py:401: SyntaxWarning: invalid escape sequence '\,'
  """


Создан DataFrame:
    Galaxy    Supernova Type     R.A.      Dec InSample              Reference
0  NGC1566  ASASSN-14ha   II  65.0059 -54.9381        ✓     Arcavietal. (2014)
1  NGC1566     SN2010el   Ia  64.9951 -54.9440        ✓  Bessell&Schmidt(2010)
2  NGC1566   SN2021aefx   Ia  64.9725 -54.9481        ✓    Valentietal. (2021)
Типы колонок: Galaxy        object
Supernova     object
Type          object
R.A.         float64
Dec          float64
InSample      object
Reference     object
dtype: object
Прочитанные данные о сверхновых:
Обрабатываю ASASSN-14ha: RA=65.0059 (тип: <class 'float'>), Dec=-54.9381 (тип: <class 'float'>), Type=II
Обрабатываю SN2010el: RA=64.9951 (тип: <class 'float'>), Dec=-54.944 (тип: <class 'float'>), Type=Ia
Обрабатываю SN2021aefx: RA=64.9725 (тип: <class 'float'>), Dec=-54.9481 (тип: <class 'float'>), Type=Ia
Изображение сохранено как: pic/trash/supernovae_map_ALMA.png


In [27]:
# Пример использования с зумом:
if __name__ == "__main__":
    plot_supernovae_on_image_zoom(
        fits_file='data/NGC1566/NGC1566_VLT_MUSE_vel_grad_xy.fits',
        sn_file='data/inputNGC1566/SN.txt',
        output_file="pic/trash/supernovae_map.png",
        zoom_size=200  # размер зуммированной области 100x100 пикселей
    )

c:\Users\kpoli\miniconda3\Lib\site-packages\astropy\wcs\wcs.py:805: FITSFixedWarning: 'datfix' made the change 'Set DATE-END to '2020-12-10T05:28:22.608' from MJD-END'.
  warnings.warn(


Создан DataFrame:
    Galaxy    Supernova Type     R.A.      Dec InSample              Reference
0  NGC1566  ASASSN-14ha   II  65.0059 -54.9381        ✓     Arcavietal. (2014)
1  NGC1566     SN2010el   Ia  64.9951 -54.9440        ✓  Bessell&Schmidt(2010)
2  NGC1566   SN2021aefx   Ia  64.9725 -54.9481        ✓    Valentietal. (2021)
Типы колонок: Galaxy        object
Supernova     object
Type          object
R.A.         float64
Dec          float64
InSample      object
Reference     object
dtype: object
Прочитанные данные о сверхновых:
    Galaxy    Supernova Type     R.A.      Dec InSample              Reference
0  NGC1566  ASASSN-14ha   II  65.0059 -54.9381        ✓     Arcavietal. (2014)
1  NGC1566     SN2010el   Ia  64.9951 -54.9440        ✓  Bessell&Schmidt(2010)
2  NGC1566   SN2021aefx   Ia  64.9725 -54.9481        ✓    Valentietal. (2021)
Обрабатываю ASASSN-14ha: X=408.3, Y=442.9
Зуммированная область сохранена: pic/trash/zoom_regions/ASASSN-14ha_zoom_200px.png
Обрабатываю SN201